Agora vamos para parte da preparação dos dados:

## 02 - Data Preparation

Com base no diagnóstico feito em `01_data_collection`, esta etapa trata os problemas 
identificados no dataset bruto para deixá-lo pronto para a modelagem. 

Principais transformações previstas:
- Correção de tipos de dados inconsistentes (ex: coluna numérica armazenada como texto) - ✅ (feito com a coluna 'TotalCharges')
- Tratamento dos valores ausentes identificados na etapa anterior - ✅ (feito com a coluna 'TotalCharges')
- Remoção de colunas sem valor preditivo (ex: identificador único do cliente) - ✅
- Codificação (encoding) das variáveis categóricas - ✅
- Análise da distribuição da variável alvo (Churn), para entender o desbalanceamento de classes - ✅

Nenhuma modelagem é realizada nesta etapa — o objetivo é exclusivamente deixar os dados 
limpos e estruturados para a etapa seguinte (`03_model_experimentation`).

In [1]:
#Aqui primeiramente preciso entender se existe alguma coluna cujo nome 
#sugere que deveria ser um número (não uma categoria).

import pandas as pd
df = pd.read_csv('data_raw/Telco-Customer-Churn.csv')
df



,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,6840-RESVB,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,...,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.5,No
7039,2234-XADUH,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.9,No
7040,4801-JZAZL,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,No
7041,8361-LTMKD,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.6,Yes


In [2]:
df.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

In [3]:
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


Percebi que na coluna 'TotalCharges' ela está como object sendo que são números no caso float64, agora nessa etapa preciso converter essa coluna.

Pesquisei o porque e encontrei isso: "O motivo mais comum desse tipo de coluna virar object mesmo parecendo números: existe algum valor "estranho" nela — geralmente um espaço em branco " " ou string vazia "" — em vez de um número de verdade ou um NaN reconhecido pelo pandas. Basta uma célula assim na coluna inteira para o pandas desistir de tratar tudo como numérico e converter a coluna inteira para texto."

In [4]:
df['TotalCharges'].describe()

count     7043
unique    6531
top       20.2
freq        11
Name: TotalCharges, dtype: object

In [5]:
# Tentando converter os valores em branco usando pd.numeric e aplicando o (error='coerce')
# forçando o pandas a converter o valor 

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

In [6]:
# Verificando se funcionou utilizando isnull() e sum().
df['TotalCharges'].isnull().sum()

np.int64(11)

**.isnull()** — percorre a coluna inteira e devolve True/False para cada uma das 7043 linhas (True onde é NaN, False onde não é). Sozinho, ainda te dá uma lista gigante, só que agora de True/False em vez dos valores originais.

**.sum()** — em Python, True vale 1 e False vale 0. Então somar essa lista de True/False dá exatamente o total de True, ou seja, o total de NaN.

In [7]:
# Dropando valores NaN para a coluna especifica
df = df.dropna(subset=['TotalCharges'])

Dropando coluna 'customerID' pois não trás nenhum valor preditivo.
Analise do motivo: um ID é único por linha, então um modelo não consegue "aprender padrão" nenhum com ele — na melhor das hipóteses é inútil, na pior atrapalha **(overfitting)**. Logo após a dropagem verifiquei com **df.columns**.

In [8]:
df = df.drop('customerID', axis=1)

In [9]:
df.columns 

Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

In [10]:
# Aqui precisa conter 7032 linhas pois o total era 7043 e eliminamos 11.
# e precisa ter 20 colunas pois eliminamos 1.
df.shape

(7032, 20)

Agora preciso descobrir a distruibuição da variavel alvo no caso 'Churn'. 
**Para entender o desbalanceamento das classes**.

In [11]:
df['Churn']

0        No
1        No
2       Yes
3        No
4       Yes
       ... 
7038     No
7039     No
7040     No
7041    Yes
7042     No
Name: Churn, Length: 7032, dtype: object

In [12]:
df['Churn'].describe()

count     7032
unique       2
top         No
freq      5163
Name: Churn, dtype: object

In [13]:
df['Churn'].value_counts()

Churn
No     5163
Yes    1869
Name: count, dtype: int64

Pesquisando descobri isso: 

**.nunique()** — responde "quantos valores diferentes existem?" (só o número total). Ex: df['Churn'].nunique() retornaria 2 (porque só existe "Yes" e "No", sem dizer quantas vezes cada um aparece).

**.value_counts()** — responde "quantas vezes cada valor aparece?". Esse é o que você quer agora para ver o balanceamento das classes:

Se quiser ver isso em proporção/porcentagem em vez de contagem bruta (às vezes mais fácil de interpretar o desbalanceamento), existe um parâmetro que converte para porcentagem — **normalize=True**

In [14]:
# Utilizando normalize='True' para transformar em porcentagem.
df['Churn'].value_counts(normalize='True')

Churn
No     0.734215
Yes    0.265785
Name: proportion, dtype: float64

Nessa etapa precisamos de algumas observações como por exemplo:

1- Separar X (atributos) e y (alvo)

2- Separar treino/teste a partir desses X e y

3- Só então aplicar o encoding, ajustado a partir do que existe no treino

Para continuarmos com as seguintes etapas precisamos instalar o scikit-learn.

In [15]:
# Separando X (atributos) e y (alvo), treino/teste a partir desses X e y
y = df['Churn']

X = df.drop('Churn', axis=1)

In [16]:
# Verificando se o alvo recebeu a coluna 'Churn'
y.shape

(7032,)

In [17]:
# Verificando se removeu a coluna 'Churn'
X.shape

(7032, 19)

In [18]:
# Importando biblioteca para teste
from sklearn.model_selection import train_test_split

### Separação em treino e teste

Antes do encoding, o dataset é dividido em conjuntos de treino e teste, para que o 
encoding seja ajustado apenas com base no que o modelo "vê" durante o treinamento — 
evitando vazamento de informação (data leakage) do conjunto de teste.

- **`test_size=...`**: define a proporção de dados reservada para teste (30%), 
  mantendo os 70% restantes para treino.
- **`random_state=...`**: fixa a "semente" de aleatoriedade do split, garantindo que 
  o mesmo resultado seja reproduzido sempre que o notebook for executado novamente.
- **`stratify=y`**: preserva a proporção original das classes de `Churn` (~73% não-churn 
  / ~27% churn) tanto no conjunto de treino quanto no de teste, evitando que o split 
  aleatório gere conjuntos com composição desbalanceada entre si.

In [19]:
# Aplicando os conceitos de treino/teste apartir de X e y.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=47, stratify=y)

conferindo com **y_train.value_counts(normalize=True)** e **y_test.value_counts(normalize=True)** se as proporções realmente ficaram parecidas nos dois conjuntos.

In [20]:
print("y_train", y_train.value_counts(normalize=True))
print("y_test", y_test.value_counts(normalize=True))

y_train Churn
No     0.734254
Yes    0.265746
Name: proportion, dtype: float64
y_test Churn
No     0.734123
Yes    0.265877
Name: proportion, dtype: float64


Preciso agora fazer o encoding que pelo o que pesquisei tem que ser a etapa final para não acontecer o temido **Vazamento de Dados (Data Leakage)**.

In [21]:
X_train.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
Index: 4922 entries, 1549 to 6493
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            4922 non-null   object 
 1   SeniorCitizen     4922 non-null   int64  
 2   Partner           4922 non-null   object 
 3   Dependents        4922 non-null   object 
 4   tenure            4922 non-null   int64  
 5   PhoneService      4922 non-null   object 
 6   MultipleLines     4922 non-null   object 
 7   InternetService   4922 non-null   object 
 8   OnlineSecurity    4922 non-null   object 
 9   OnlineBackup      4922 non-null   object 
 10  DeviceProtection  4922 non-null   object 
 11  TechSupport       4922 non-null   object 
 12  StreamingTV       4922 non-null   object 
 13  StreamingMovies   4922 non-null   object 
 14  Contract          4922 non-null   object 
 15  PaperlessBilling  4922 non-null   object 
 16  PaymentMethod     4922 non-null   object 
 1

In [22]:
for col in X_train.select_dtypes(include='object').columns:
    print(col, X_train[col].nunique())

gender 2
Partner 2
Dependents 2
PhoneService 2
MultipleLines 3
InternetService 3
OnlineSecurity 3
OnlineBackup 3
DeviceProtection 3
TechSupport 3
StreamingTV 3
StreamingMovies 3
Contract 3
PaperlessBilling 2
PaymentMethod 4


Pesquisando descobri que o scikit-learn possui uma ferramente chamada ColumnTransformer.

Para que ele funcione, a primeira coisa que precisamos fornecer a ele são duas listas simples:

Uma lista contendo apenas os nomes das colunas **numéricas**.

Uma lista contendo apenas os nomes das colunas **categóricas (de texto)**.

In [23]:
# Atribuindo a uma variavel apenas as colunas catégoricas (de texto)
colunas_categoricas = X_train.select_dtypes(include='object').columns.to_list()
# Atribuindo as colunas numéricas.
colunas_numericas = X_train.select_dtypes(include='number').columns.to_list()

# Verificando se funcionou as atribuições
print("Colunas Númericas:", colunas_numericas)
print("Colunas de texto:", colunas_categoricas)

Colunas Númericas: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
Colunas de texto: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


# Transformação de Variáveis (Encoding)

Nesta etapa, preparamos os dados para os algoritmos matemáticos de Machine Learning, que não processam texto. Para isso, utilizamos duas ferramentas essenciais do Scikit-Learn:

**OneHotEncoder**: Transforma nossas colunas categóricas (texto) em colunas binárias (0 ou 1). Isso é crucial para não criar falsas hierarquias matemáticas (ex: o modelo achar que um contrato de 2 anos vale o "dobro" de um contrato de 1 ano).

**ColumnTransformer**: Atua como o "maestro" do nosso pré-processamento. Ele nos permite aplicar regras diferentes de forma simultânea (ex: aplicar o OneHotEncoder apenas nas colunas de texto e deixar as colunas numéricas passarem intactas).

In [33]:
# Importando as bibliotecas necessarias para o encoding
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

Se precisarmos que as colunas numéricas passem intactas pelo nosso "ColumnTransformer" por enquanto, usamos o termo técnico no Scikit-Learn para "deixar passar" que é 'passthrough'.

Como não sabia como continuar, pesquisando encontrei isso:

Agora nós criamos uma variável (por exemplo, **pre_processador**) e chamamos o **ColumnTransformer**. Dentro dele, passamos uma lista de tuplas. **Cada tupla tem 3 partes**:

1- O nome que você quer dar para aquele passo **(ex: 'categorico')**.

2- A ferramenta que você vai usar **(ex: OneHotEncoder())**.

3- A lista de colunas onde isso vai acontecer **(a nossa variável colunas_categoricas)**.

Sabendo que a ferramenta do Scikit-Learn se chama **OneHotEncoder(drop='first')** — (esse **drop='first'** é um truque para evitar dados redundantes, chamado de armadilha da variável **dummy**)

Sobre o **(drop='first')**: O drop='first' dentro do OneHotEncoder é um escudo de proteção estatística. Por exemplo, na coluna `Partner` (Yes/No), se soubermos que a pessoa não tem 'Yes', sabemos automaticamente que ela é 'No' — não precisamos das duas colunas para carregar essa informação. Se mantivermos todas as colunas, a matemática cria dados redundantes. O drop='first' apaga a primeira opção e deixa o modelo deduzir por eliminação.

Obs: Após analisar quais modelos meu professor pediu no projeto para utilizar na fase de experimentação preliminar, sendo eles **(Regressão Logística, Árvore de Decisão, Random Forest, KNN ou SVM)**, pesquisando encontrei algumas informações e aprendi que daria problemas em alguns modelos caso não fizesse a normalização de algumas colunas como tenure, MonthlyCharges e TotalCharges.

Após a pesquisa fiz uma alteração na linha 34 do código colocando mais uma tupla dentro do `ColumnTransformer` utilizando a biblioteca `StandardScaler` do `sklearn.preprocessing`, ela transforma cada coluna numérica para ter média 0 e desvio padrão 1.

In [34]:
# Criando nosso ColumnTransformer
pre_processador = ColumnTransformer(
transformers=[('categóricas', OneHotEncoder(drop='first'), colunas_categoricas)
                ,
                ('numéricas', StandardScaler(), colunas_numericas)]
)

Agora começando a fazer a máquina aprender abaixo vou deixar o que pesquisei e encontrei sobre:

Precisamos analisar os comandos porque entender essa diferença é um dos maiores pilares do Machine Learning em Python:

**.fit() (Aprender)**: Faz o modelo olhar para os dados, memorizar quais categorias existem e criar a regra matemática. Mas ele não altera a tabela.

**.transform() (Aplicar)**: Pega a regra que foi memorizada no passo anterior e aplica na tabela, modificando os dados de fato.

Como no X_train nós precisamos fazer as duas coisas ao mesmo tempo (ensinar as regras das categorias para o nosso "maestro"(ColumnTransformer) e já converter o texto em números), o Scikit-Learn criou um "atalho" que junta essas duas palavras: o **.fit_transform()**.

In [35]:
# Treinando o "maestro"(ColumnTransformer) E aplicando as transformações no X_train
X_train_processado = pre_processador.fit_transform(X_train)

### Aplicando o encoding no conjunto de teste

Usamos `transform` (sem `fit`) para reaplicar no teste as mesmas regras de encoding 
já aprendidas com o `X_train`, evitando data leakage — o teste deve ser processado 
com as regras do treino, nunca aprender regras próprias.

In [36]:
#Aplicando transformação no X_test
X_test_processado = pre_processador.transform(X_test)

Depois disso, um bom hábito de conferência: dá uma olhada em `X_train_processado.shape` e `X_test_processado.shape` — as duas devem ter o **mesmo número de colunas** (só o número de linhas deve diferir, já que são conjuntos de tamanhos diferentes). Se o número de colunas bater, é sinal de que o **encoding** foi consistente entre os dois conjuntos.

In [37]:
X_train_processado.shape

(4922, 30)

In [38]:
X_test_processado.shape

(2110, 30)

### Resumo da Etapa 2 — Data Preparation

✅ **`TotalCharges`**: convertida de `object` para numérica via `pd.to_numeric(errors='coerce')`; 
  11 valores ausentes identificados e removidos (`dropna`) — dataset passou de 7043 para 7032 linhas.

✅ **`customerID`**: removida por não ter valor preditivo (identificador único por cliente) — 
  de 21 para 20 colunas.

✅ **`Churn` (alvo)**: distribuição de ~73% (não-churn) / 27% (churn) — desbalanceamento 
  moderado, considerado na escolha de métricas para a etapa 3.

✅ **Split treino/teste**: `test_size=0.3`, `random_state=47`, `stratify=y` — proporção 
  de classes preservada em ambos os conjuntos.

✅ **Encoding**: `OneHotEncoder(drop='first')` via `ColumnTransformer`, ajustado apenas 
  no treino (`fit_transform`) e reaplicado no teste (`transform`), evitando data leakage.

✅ **Normalização**: `StandardScaler` aplicado às colunas numéricas (`tenure`, 
  `MonthlyCharges`, `TotalCharges`), no mesmo `ColumnTransformer` do encoding.
  
✅ **Resultado final**: `X_train_processado` e `X_test_processado` com 30 colunas cada, 
  prontos para a Etapa 3.